In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import datetime
import tqdm

print(datetime.datetime.now().isoformat())

2025-06-17T19:02:55.806993


In [3]:
import bioacoustics_model_zoo as bmz

2025-06-17 19:10:08.400853: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-06-17 19:10:27.460505: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-06-17 19:10:28.844310: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-06-17 19:10:29.816025: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-06-17 19:10:32.174964: I tensorflow/core/platform/cpu_feature_guar

In [4]:
from pathlib import Path

audio = sorted(Path("~/scratch/birdclef/raw/birdclef-2024/train_audio/asbfly").expanduser().glob("*.ogg"))[
    :2
]
audio

[PosixPath('/storage/home/hcoda1/7/acheung46/scratch/birdclef/raw/birdclef-2024/train_audio/asbfly/XC134896.ogg'),
 PosixPath('/storage/home/hcoda1/7/acheung46/scratch/birdclef/raw/birdclef-2024/train_audio/asbfly/XC164848.ogg')]

In [4]:
perch = bmz.list_models()["Perch"]()
perch

/storage/scratch1/7/acheung46/birdclef/.venv/lib64/python3.9/site-packages/tensorflow_hub/__init__.py:61: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  from pkg_resources import parse_version
/storage/scratch1/7/acheung46/birdclef/.venv/lib64/python3.9/site-packages/opensoundscape/ml/cnn.py:599: UserWarning: 
                    This architecture is not listed in opensoundscape.ml.cnn_architectures.ARCH_DICT.
                    It will not be available for loading after saving the model with .save() (unless using pickle=True). 
                    To make it re-loadable, define a function that generates the architecture from arguments: (n_classes, n_channels) 
                    then use opensoundscape.ml.cnn_architectures.register_architecture() to register the generating function.

                    The function can also set the returned object's .constructor_name to the registered string key in ARCH_DICT


Perch(
  (network): MLPClassifier(
    (hidden_layers): Sequential()
    (classifier): Linear(in_features=1280, out_features=10932, bias=True)
  )
  (loss_fn): BCEWithLogitsLoss_hot()
)

In [6]:
birdnet = bmz.list_models()["BirdNET"]()
birdnet

File BirdNET_GLOBAL_6K_V2.4_Labels_af.txt already exists; skipping download.
downloading model from URL...
File BirdNET_GLOBAL_6K_V2.4_Model_FP16.tflite already exists; skipping download.


/storage/scratch1/7/acheung46/birdclef/.venv/lib64/python3.9/site-packages/opensoundscape/ml/cnn.py:599: UserWarning: 
                    This architecture is not listed in opensoundscape.ml.cnn_architectures.ARCH_DICT.
                    It will not be available for loading after saving the model with .save() (unless using pickle=True). 
                    To make it re-loadable, define a function that generates the architecture from arguments: (n_classes, n_channels) 
                    then use opensoundscape.ml.cnn_architectures.register_architecture() to register the generating function.

                    The function can also set the returned object's .constructor_name to the registered string key in ARCH_DICT
                    to avoid this warning and ensure it is reloaded correctly by opensoundscape.ml.load_model().

                    See opensoundscape.ml.cnn_architectures module for examples of constructor functions
                    
  warnings.warn(
/storage/s

BirdNET(
  (network): MLPClassifier(
    (hidden_layers): Sequential()
    (classifier): Linear(in_features=1024, out_features=6522, bias=True)
  )
  (loss_fn): BCEWithLogitsLoss_hot()
)

In [7]:
# NOTE: there is a warmup period for the model...
birdnet.predict(audio[0:1])
%time _ = birdnet.predict(audio)

  0%|          | 0/9 [00:00<?, ?it/s]

  0%|          | 0/14 [00:00<?, ?it/s]

CPU times: user 1.5 s, sys: 2.91 ms, total: 1.51 s
Wall time: 1.55 s


In [13]:
%time _ = birdnet.predict(audio, clip_step=1)

  0%|          | 0/38 [00:00<?, ?it/s]

CPU times: user 4.04 s, sys: 23.4 ms, total: 4.06 s
Wall time: 4.05 s


In [8]:
%time _ = birdnet.embed(audio, return_preds=False)

  0%|          | 0/14 [00:00<?, ?it/s]

CPU times: user 1.49 s, sys: 5.5 ms, total: 1.49 s
Wall time: 1.5 s


In [9]:
%time _ = birdnet.embed(audio, return_preds=False, clip_step=1)

  0%|          | 0/38 [00:00<?, ?it/s]

CPU times: user 3.99 s, sys: 24.6 ms, total: 4.01 s
Wall time: 4.03 s


In [8]:
# NOTE: there is a warmup period for the model...
perch.predict(audio[0:1])
%time _ = perch.predict(audio)

  0%|          | 0/5 [00:00<?, ?it/s]

I0000 00:00:1750131539.825091 2836388 service.cc:146] XLA service 0x5555746a3080 initialized for platform Host (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1750131539.967671 2836388 service.cc:154]   StreamExecutor device (0): Host, Default Version
2025-06-16 23:39:07.049569: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
W0000 00:00:1750131553.808833 2836388 assert_op.cc:38] Ignoring Assert operator jax2tf_infer_fn_/assert_equal_1/Assert/AssertGuard/Assert
I0000 00:00:1750131577.531276 2836388 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


  0%|          | 0/8 [00:00<?, ?it/s]

CPU times: user 11.3 s, sys: 23 ms, total: 11.3 s
Wall time: 10.4 s


In [73]:
from birdclef.kaggle.compile import load_tflite_interpreter, run_perch_tflite
import tensorflow as tf
from contexttimer import Timer

# Path to the TFLite model
perch_tflite_path = Path("~/scratch/birdclef/models/2025/v1/Perch/torch-linear-v1/perch.tflite").expanduser()

# Load the TFLite interpreter
perch_interpreter = load_tflite_interpreter(perch_tflite_path)

# First run to warm up
perch_dataloader = perch.predict_dataloader(audio[0:1])
_ = run_perch_tflite(perch_interpreter, perch_dataloader)

# Measure execution time
def run_perch():
    perch_dataloader = perch.predict_dataloader(audio)
    return run_perch_tflite(perch_interpreter, perch_dataloader)

%time perch_out = run_perch()

2025-06-17 01:15:34.816675: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_inter_op_parallelism which is not in the op definition: Op<name=Transpose; signature=x:T, perm:Tperm -> y:T; attr=T:type; attr=Tperm:type,default=DT_INT32,allowed=[DT_INT32, DT_INT64]> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node Transpose}}


CPU times: user 846 ms, sys: 6.69 ms, total: 852 ms
Wall time: 847 ms


In [43]:
perch_out

0     \
file                                               start_time end_time             
/storage/home/hcoda1/7/acheung46/scratch/birdcl... 0.0        5.0      -0.042605   
                                                   5.0        10.0      0.043925   
                                                   10.0       15.0      0.191889   
                                                   15.0       20.0      0.078761   
                                                   20.0       25.0     -0.008164   
/storage/home/hcoda1/7/acheung46/scratch/birdcl... 0.0        5.0      -0.014053   
                                                   5.0        10.0      0.100804   
                                                   10.0       15.0      0.064618   

                                                                            1     \
file                                               start_time end_time             
/storage/home/hcoda1/7/acheung46/scratch/birdcl... 0.0        5.0      -0.141005   
                                                   5.0        10.0     -0.158573   
                                                   10.0       15.0     -0.165586   
                                                   15.0       20.0     -0.127910   
                                                   20.0       25.0     -0.166585   
/storage/home/hcoda1/7/acheung46/scratch/birdcl... 0.0        5.0      -0.047585   
                                                   5.0        10.0     -0.103619   
                                                   10.0       15.0     -0.125703   

                                                                            2     \
file                                               start_time end_time             
/storage/home/hcoda1/7/acheung46/scratch/birdcl... 0.0        5.0       0.025648   
                                                   5.0        10.0      0.057496   
                                                   10.0       15.0      0.032176   
                                                   15.0       20.0      0.067172   
                                                   20.0       25.0      0.077055   
/storage/home/hcoda1/7/acheung46/scratch/birdcl... 0.0        5.0       0.023723   
                                                   5.0        10.0      0.048283   
                                                   10.0       15.0      0.040848   

                                                                            3     \
file                                               start_time end_time             
/storage/home/hcoda1/7/acheung46/scratch/birdcl... 0.0        5.0       0.041677   
                                                   5.0        10.0      0.006564   
                                                   10.0       15.0      0.041329   
                                                   15.0       20.0     -0.001162   
                                                   20.0       25.0      0.020314   
/storage/home/hcoda1/7/acheung46/scratch/birdcl... 0.0        5.0      -0.023687   
                                                   5.0        10.0     -0.017900   
                                                   10.0       15.0     -0.039507   

                                                                            4     \
file                                               start_time end_time             
/storage/home/hcoda1/7/acheung46/scratch/birdcl... 0.0        5.0       0.219057   
                                                   5.0        10.0      0.133760   
                                                   10.0       15.0      0.241032   
                                                   15.0       20.0      0.362048   
                                                   20.0       25.0      0.305957   
/storage/home/hcoda1/7/acheung46/scratch/birdcl... 0.0        5.0       0.064599   
                                        

In [40]:
perch_out.shape

(8, 1280)

In [10]:
perch_interpreter.get_input_details()

[{'name': 'serving_default_inputs:0',
  'index': 0,
  'shape': array([     1, 160000], dtype=int32),
  'shape_signature': array([    -1, 160000], dtype=int32),
  'dtype': numpy.float32,
  'quantization': (0.0, 0),
  'quantization_parameters': {'scales': array([], dtype=float32),
   'zero_points': array([], dtype=int32),
   'quantized_dimension': 0},
  'sparsity_parameters': {}}]

In [11]:
perch_interpreter.get_output_details()

[{'name': 'StatefulPartitionedCall:2',
  'index': 384,
  'shape': array([  1, 500, 160], dtype=int32),
  'shape_signature': array([ -1, 500, 160], dtype=int32),
  'dtype': numpy.float32,
  'quantization': (0.0, 0),
  'quantization_parameters': {'scales': array([], dtype=float32),
   'zero_points': array([], dtype=int32),
   'quantized_dimension': 0},
  'sparsity_parameters': {}},
 {'name': 'StatefulPartitionedCall:0',
  'index': 828,
  'shape': array([   1, 1280], dtype=int32),
  'shape_signature': array([  -1, 1280], dtype=int32),
  'dtype': numpy.float32,
  'quantization': (0.0, 0),
  'quantization_parameters': {'scales': array([], dtype=float32),
   'zero_points': array([], dtype=int32),
   'quantized_dimension': 0},
  'sparsity_parameters': {}},
 {'name': 'StatefulPartitionedCall:3',
  'index': 832,
  'shape': array([   1, 2333], dtype=int32),
  'shape_signature': array([  -1, 2333], dtype=int32),
  'dtype': numpy.float32,
  'quantization': (0.0, 0),
  'quantization_parameters': {'

In [31]:
from birdclef.kaggle.compile import load_tflite_interpreter
import tensorflow as tf
from contexttimer import Timer

# Path to the TFLite model
birdnet_tflite_path = Path("~/scratch/birdclef/models/2025/v1/BirdNET/torch-linear-v1/birdnet.tflite").expanduser()

# Load the TFLite interpreter
birdnet_interpreter = load_tflite_interpreter(birdnet_tflite_path)

In [33]:
birdnet_input_details = birdnet_interpreter.get_input_details()
birdnet_output_details = birdnet_interpreter.get_output_details()

In [14]:
birdnet_input_details

[{'name': 'INPUT',
  'index': 0,
  'shape': array([     1, 144000], dtype=int32),
  'shape_signature': array([    -1, 144000], dtype=int32),
  'dtype': numpy.float32,
  'quantization': (0.0, 0),
  'quantization_parameters': {'scales': array([], dtype=float32),
   'zero_points': array([], dtype=int32),
   'quantized_dimension': 0},
  'sparsity_parameters': {}}]

In [15]:
birdnet_output_details

[{'name': 'Identity',
  'index': 546,
  'shape': array([   1, 6522], dtype=int32),
  'shape_signature': array([  -1, 6522], dtype=int32),
  'dtype': numpy.float32,
  'quantization': (0.0, 0),
  'quantization_parameters': {'scales': array([], dtype=float32),
   'zero_points': array([], dtype=int32),
   'quantized_dimension': 0},
  'sparsity_parameters': {}}]

In [34]:
birdnet_dataloader = birdnet.predict_dataloader(audio, clip_step=1)

res = []
for batch in birdnet_dataloader:
    birdnet_interpreter.set_tensor(birdnet_input_details[0]['index'], batch[0])
    birdnet_interpreter.invoke()
    output_data = birdnet_interpreter.get_tensor(birdnet_output_details[0]['index'])
    res.append(output_data)

res[0].shape


(1, 6522)

In [ ]:
birdnet_interpreter = load_tflite_interpreter(birdnet_tflite_path)
# birdnet_interpreter.allocate_tensors()

birdnet_dataloader = birdnet.predict_dataloader(audio, clip_step=1)

res = []
for batch in birdnet_dataloader:
    birdnet_input_details = birdnet_interpreter.get_input_details()[0]
    input_layer_idx = birdnet_input_details['index']
    birdnet_output_details = birdnet_interpreter.get_output_details()[0]
    embedding_idx = birdnet_output_details['index'] - 1
    
    batch = batch[0]
    birdnet_interpreter.resize_tensor_input(
        input_layer_idx,
        [len(batch), *batch[0].shape]
    )
    birdnet_interpreter.allocate_tensors()
    birdnet_interpreter.set_tensor(input_layer_idx, np.array(batch, dtype=np.float32))
    birdnet_interpreter.invoke()
    output_data = birdnet_interpreter.get_tensor(embedding_idx)
    
    res.append(output_data)
res[0].shape


(1, 1024)

In [69]:
import numpy as np
import pandas as pd

birdnet_out_df = pd.DataFrame(
    data=np.stack(res).squeeze(),
    index=birdnet_dataloader.dataset.dataset.label_df.index,
)
birdnet_out_df

0     \
file                                               start_time end_time             
/storage/home/hcoda1/7/acheung46/scratch/birdcl... 0.0        3.0       0.000000   
                                                   1.0        4.0       0.000000   
                                                   2.0        5.0       0.334714   
                                                   3.0        6.0       0.079557   
                                                   4.0        7.0       0.000000   
                                                   5.0        8.0       0.000000   
                                                   6.0        9.0       0.000000   
                                                   7.0        10.0      0.083778   
                                                   8.0        11.0      0.000000   
                                                   9.0        12.0      0.000000   
                                                   10.0       13.0      0.000000   
                                                   11.0       14.0      0.000000   
                                                   12.0       15.0      0.036325   
                                                   13.0       16.0      0.000000   
                                                   14.0       17.0      0.000000   
                                                   15.0       18.0      0.000000   
                                                   16.0       19.0      0.056075   
                                                   17.0       20.0      0.000000   
                                                   18.0       21.0      0.007163   
                                                   19.0       22.0      0.000000   
                                                   20.0       23.0      0.000000   
                                                   21.0       24.0      0.226059   
                                                   22.0       25.0      0.110504   
                                                   23.0       26.0      0.000000   
                                                   24.0       27.0      0.000000   
/storage/home/hcoda1/7/acheung46/scratch/birdcl... 0.0        3.0       0.047440   
                                                   1.0        4.0       0.281672   
                                                   2.0        5.0       0.025264   
                                                   3.0        6.0       0.090743   
                                                   4.0        7.0       0.131011   
                                                   5.0        8.0       0.000000   
                                                   6.0        9.0       0.000000   
                                                   7.0        10.0      0.000000   
                                                   8.0        11.0      0.000000   
                                                   9.0        12.0      0.000000   
                                                   10.0       13.0      0.000000   
                                                   11.0       14.0      0.000000   
                                                   12.0       15.0      0.000000   

                                                                            1     \
file                                               start_time end_time             
/storage/home/hcoda1/7/acheung46/scratch/birdcl... 0.0        3.0       1.776116   
                                                   1.0        4.0       2.351565   
                                                   2.0        5.0       2.474020   
                                                   3.0        6.0       2.158225   
                                                   4.0        7.0       2.082779   
                                                   5.0        8.0       2.529492   
                                           

In [65]:
bmz_birdnet_out = birdnet.embed(audio, clip_step=1)
print(bmz_birdnet_out.shape)
pd.DataFrame(
    data=bmz_birdnet_out.to_numpy(),
    index=birdnet_dataloader.dataset.dataset.label_df.index,
)

  0%|          | 0/38 [00:00<?, ?it/s]

(38, 1024)


0     \
file                                               start_time end_time             
/storage/home/hcoda1/7/acheung46/scratch/birdcl... 0.0        3.0       0.000000   
                                                   1.0        4.0       0.000000   
                                                   2.0        5.0       0.334715   
                                                   3.0        6.0       0.079556   
                                                   4.0        7.0       0.000000   
                                                   5.0        8.0       0.000000   
                                                   6.0        9.0       0.000000   
                                                   7.0        10.0      0.083778   
                                                   8.0        11.0      0.000000   
                                                   9.0        12.0      0.000000   
                                                   10.0       13.0      0.000000   
                                                   11.0       14.0      0.000000   
                                                   12.0       15.0      0.036325   
                                                   13.0       16.0      0.000000   
                                                   14.0       17.0      0.000000   
                                                   15.0       18.0      0.000000   
                                                   16.0       19.0      0.056075   
                                                   17.0       20.0      0.000000   
                                                   18.0       21.0      0.007163   
                                                   19.0       22.0      0.000000   
                                                   20.0       23.0      0.000000   
                                                   21.0       24.0      0.226059   
                                                   22.0       25.0      0.110503   
                                                   23.0       26.0      0.000000   
                                                   24.0       27.0      0.000000   
/storage/home/hcoda1/7/acheung46/scratch/birdcl... 0.0        3.0       0.047439   
                                                   1.0        4.0       0.281672   
                                                   2.0        5.0       0.025264   
                                                   3.0        6.0       0.090742   
                                                   4.0        7.0       0.131011   
                                                   5.0        8.0       0.000000   
                                                   6.0        9.0       0.000000   
                                                   7.0        10.0      0.000000   
                                                   8.0        11.0      0.000000   
                                                   9.0        12.0      0.000000   
                                                   10.0       13.0      0.000000   
                                                   11.0       14.0      0.000000   
                                                   12.0       15.0      0.000000   

                                                                            1     \
file                                               start_time end_time             
/storage/home/hcoda1/7/acheung46/scratch/birdcl... 0.0        3.0       1.776114   
                                                   1.0        4.0       2.351564   
                                                   2.0        5.0       2.474020   
                                                   3.0        6.0       2.158225   
                                                   4.0        7.0       2.082778   
                                                   5.0        8.0       2.529491   
                                           

In [66]:
from birdclef.kaggle.compile import run_birdnet_tflite

# birdnet_dataloader = birdnet.predict_dataloader(audio[0:1])
# _ = run_birdnet_tflite(birdnet_interpreter, birdnet_dataloader)

def run_birdnet(interpreter, clip_step=None):
    birdnet_dataloader = birdnet.predict_dataloader(audio, clip_step=clip_step)
    return run_birdnet_tflite(interpreter, birdnet_dataloader)

%time _ = run_birdnet(birdnet_interpreter)
%time _ = run_birdnet(birdnet_interpreter, clip_step=1)

CPU times: user 1.26 s, sys: 8.78 ms, total: 1.27 s
Wall time: 1.28 s
CPU times: user 3.39 s, sys: 33.8 ms, total: 3.42 s
Wall time: 3.46 s


In [67]:
birdnet_interpreter_multi = load_tflite_interpreter(birdnet_tflite_path, num_threads=8)

%time _ = run_birdnet(birdnet_interpreter_multi)
%time _ = run_birdnet(birdnet_interpreter_multi, clip_step=1)

CPU times: user 1.84 s, sys: 381 ms, total: 2.23 s
Wall time: 1.7 s
CPU times: user 4.21 s, sys: 162 ms, total: 4.37 s
Wall time: 2.59 s


### Without experimental_preserve_all_tensors

In [139]:
from birdclef.kaggle.compile import run_birdnet_tflite

def _load_tflite_interpreter(model_path: Path, num_threads: int = None):
    """Load a TFLite interpreter for the given model path."""
    interpreter = tf.lite.Interpreter(
        num_threads=num_threads,
        model_path=Path(model_path).expanduser().as_posix(),
    )
    interpreter.allocate_tensors()
    return interpreter

def _run_birdnet_tflite(interpreter, dataloader) -> pd.DataFrame:
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()
    res = []
    for batch in dataloader:
        interpreter.set_tensor(input_details[0]["index"], batch[0])
        interpreter.invoke()
        output_data = interpreter.get_tensor(output_details[0]["index"])
        res.append(output_data)
    return pd.DataFrame(
        data=np.stack(res).squeeze(),
        index=dataloader.dataset.dataset.label_df.index,
    )

def _run_birdnet(interpreter, clip_step=None):
    birdnet_dataloader = birdnet.predict_dataloader(audio, clip_step=clip_step)
    return _run_birdnet_tflite(interpreter, birdnet_dataloader)

In [140]:
birdnet_interpreter_no_preserve = _load_tflite_interpreter(birdnet_tflite_path, num_threads=8)

%time _ = _run_birdnet(birdnet_interpreter_no_preserve)
%time _ = _run_birdnet(birdnet_interpreter_no_preserve, clip_step=1)

CPU times: user 1.11 s, sys: 47.8 ms, total: 1.15 s
Wall time: 679 ms
CPU times: user 3.81 s, sys: 33.8 ms, total: 3.84 s
Wall time: 1.87 s


In [141]:
birdnet_interpreter_no_preserve_multi = _load_tflite_interpreter(birdnet_tflite_path, num_threads=8)

%time _ = _run_birdnet(birdnet_interpreter_no_preserve_multi)
%time _ = _run_birdnet(birdnet_interpreter_no_preserve_multi, clip_step=1)

CPU times: user 1.91 s, sys: 19.6 ms, total: 1.93 s
Wall time: 790 ms
CPU times: user 4.14 s, sys: 36.1 ms, total: 4.18 s
Wall time: 1.9 s


In [134]:
from pathlib import Path
import json
from birdclef.config import model_config
from birdclef.torch.model import LinearClassifier

model_path = Path("~/scratch/birdclef/models/2025/v1/BirdNET/torch-linear-v1/").expanduser()

label_to_index = json.loads((model_path / "label_to_idx.json").read_text())
checkpoint = list(model_path.glob("checkpoints/*.ckpt"))[0]
classifier = LinearClassifier.load_from_checkpoint(
    checkpoint.as_posix(),
    input_dim=model_config["BirdNET"]["embed_size"],
    num_classes=len(label_to_index),
)
classifier.eval()

LinearClassifier(
  (model): Sequential(
    (0): Linear(in_features=1024, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=205, bias=True)
  )
  (loss_fn): CrossEntropyLoss()
)

In [135]:
import polars as pl
import torch

# convert the column "file" to a string
df = birdnet_out_df.reset_index()
df["file"] = df["file"].astype(str)
df = pl.from_pandas(df)
df.head()

file,start_time,end_time,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,…,987,988,989,990,991,992,993,994,995,996,997,998,999,1000,1001,1002,1003,1004,1005,1006,1007,1008,1009,1010,1011,1012,1013,1014,1015,1016,1017,1018,1019,1020,1021,1022,1023
str,f64,f64,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,…,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32
"""/storage/home/hcoda1/7/acheung…",0.0,3.0,0.0,1.776116,1.586924,0.0,0.756652,0.33903,0.013997,0.653717,0.0,0.0,0.0,0.037036,0.039334,0.201803,0.038165,0.881205,0.098853,1.02096,0.0,0.933884,0.322899,0.213252,0.984052,0.0,0.272441,0.10101,0.0,0.749055,0.731103,0.263828,0.665999,0.096504,1.187314,0.676258,…,0.025008,0.392178,0.0,0.103531,0.069154,0.0,0.0,0.558831,0.13408,0.833962,0.439207,0.410874,0.143482,0.0,0.0,0.508324,0.213104,0.40948,0.073897,0.0,0.054516,0.0,0.0,1.405881,0.501308,0.248858,0.740999,0.666299,0.123103,0.560407,0.820016,0.0,0.0,1.088349,0.308457,0.792822,1.315303
"""/storage/home/hcoda1/7/acheung…",1.0,4.0,0.0,2.351565,1.668995,0.0,0.869696,0.481594,0.131164,0.68934,0.096475,0.0,0.0,0.062434,0.060956,0.171884,0.0,1.26068,0.392044,0.926676,0.052018,1.011633,0.0,0.054648,0.617735,0.186856,0.563554,0.146584,0.0,0.630087,0.896474,0.147217,1.015036,0.078888,1.247754,1.552428,…,0.0,0.668515,0.0,0.179666,0.006797,0.0,0.0,0.210543,0.027931,0.482153,0.451544,0.204613,0.147526,0.0,0.0,0.27946,0.391577,0.239493,0.068529,0.0,0.130485,0.120041,0.0,1.594263,0.911198,0.152486,0.739819,1.254766,0.490242,0.705703,1.263876,0.01615,0.0,1.272554,0.420784,1.083935,1.059069
"""/storage/home/hcoda1/7/acheung…",2.0,5.0,0.334714,2.47402,1.460096,0.0,0.876815,0.453922,0.0,0.391301,0.0,0.0,0.0,0.085405,0.406965,0.166353,0.0,1.155489,0.286686,0.415053,0.051463,1.018661,0.172012,0.0,0.348614,0.0,1.257701,0.407575,0.0,0.026595,1.290908,0.559517,0.895577,0.366825,0.401949,1.275324,…,0.097596,0.80291,0.363927,0.750088,0.0,0.0,0.0,0.824933,0.0,0.400426,0.365139,0.07626,0.013322,0.0,0.0,0.0,0.519333,1.272686,0.0,0.0,0.0,0.532389,0.0,0.550662,0.263516,0.033515,0.611871,0.880727,0.304514,1.195917,0.129151,0.04601,0.0,1.219458,0.210007,1.754441,0.529761
"""/storage/home/hcoda1/7/acheung…",3.0,6.0,0.079557,2.158225,0.970361,0.0,1.015812,0.547688,0.341653,0.155102,0.114957,0.0,0.0,0.586919,0.155601,0.591243,0.03109,1.032308,0.790121,0.730858,0.0,1.297465,0.000998,0.125733,0.107348,0.041562,0.780195,1.038781,0.0,0.243233,0.53852,0.0,1.243877,0.86334,0.656169,1.564929,…,0.0,0.334183,0.727118,0.172298,0.058998,0.038082,0.121399,0.061113,0.379611,0.092362,0.522844,0.0,0.38986,0.0,0.0,0.050778,0.268432,0.518953,0.280309,0.0,0.191015,0.092905,0.01835,0.073994,1.36536,0.149721,0.120322,0.396275,0.507074,0.182179,0.191319,0.325938,0.0,1.940921,0.761787,0.853025,0.0
"""/storage/home/hcoda1/7/acheung…",4.0,7.0,0.0,2.082779,1.094857,0.0,0.778769,0.349799,0.144341,0.579971,0.013571,0.0,0.0,0.05166,0.502043,0.544351,0.0,0.909057,0.446395,0.725157,0.078493,1.50786,0.417534,0.0,0.474653,0.300856,0.715112,0.352991,0.0,1.191465,1.067876,0.223735,1.495245,0.504632,0.70172,0.885171,…,0.0,0.658137,0.0,0.119978,0.070171,0.0,0.0,0.484041,0.0,0.170101,0.103452,0.0,0.567871,0.0,0.0,0.442525,0.859334,0.196393,0.018195,0.383672,0.179479,0.08112,0.0,0.800949,0.410062,0.420333,1.004975,0.740951,0.243811,0.519912,0.418317,0.002026,0.0,1.497738,0.214118,0.992445,1.318701


In [136]:
interval_length = 5
df = df.with_columns(
    ((pl.col("start_time") + pl.col("end_time")) / 2 / interval_length)
    .cast(pl.Int64)
    .alias("interval")
)
df.head()

file,start_time,end_time,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,…,988,989,990,991,992,993,994,995,996,997,998,999,1000,1001,1002,1003,1004,1005,1006,1007,1008,1009,1010,1011,1012,1013,1014,1015,1016,1017,1018,1019,1020,1021,1022,1023,interval
str,f64,f64,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,…,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,i64
"""/storage/home/hcoda1/7/acheung…",0.0,3.0,0.0,1.776116,1.586924,0.0,0.756652,0.33903,0.013997,0.653717,0.0,0.0,0.0,0.037036,0.039334,0.201803,0.038165,0.881205,0.098853,1.02096,0.0,0.933884,0.322899,0.213252,0.984052,0.0,0.272441,0.10101,0.0,0.749055,0.731103,0.263828,0.665999,0.096504,1.187314,0.676258,…,0.392178,0.0,0.103531,0.069154,0.0,0.0,0.558831,0.13408,0.833962,0.439207,0.410874,0.143482,0.0,0.0,0.508324,0.213104,0.40948,0.073897,0.0,0.054516,0.0,0.0,1.405881,0.501308,0.248858,0.740999,0.666299,0.123103,0.560407,0.820016,0.0,0.0,1.088349,0.308457,0.792822,1.315303,0
"""/storage/home/hcoda1/7/acheung…",1.0,4.0,0.0,2.351565,1.668995,0.0,0.869696,0.481594,0.131164,0.68934,0.096475,0.0,0.0,0.062434,0.060956,0.171884,0.0,1.26068,0.392044,0.926676,0.052018,1.011633,0.0,0.054648,0.617735,0.186856,0.563554,0.146584,0.0,0.630087,0.896474,0.147217,1.015036,0.078888,1.247754,1.552428,…,0.668515,0.0,0.179666,0.006797,0.0,0.0,0.210543,0.027931,0.482153,0.451544,0.204613,0.147526,0.0,0.0,0.27946,0.391577,0.239493,0.068529,0.0,0.130485,0.120041,0.0,1.594263,0.911198,0.152486,0.739819,1.254766,0.490242,0.705703,1.263876,0.01615,0.0,1.272554,0.420784,1.083935,1.059069,0
"""/storage/home/hcoda1/7/acheung…",2.0,5.0,0.334714,2.47402,1.460096,0.0,0.876815,0.453922,0.0,0.391301,0.0,0.0,0.0,0.085405,0.406965,0.166353,0.0,1.155489,0.286686,0.415053,0.051463,1.018661,0.172012,0.0,0.348614,0.0,1.257701,0.407575,0.0,0.026595,1.290908,0.559517,0.895577,0.366825,0.401949,1.275324,…,0.80291,0.363927,0.750088,0.0,0.0,0.0,0.824933,0.0,0.400426,0.365139,0.07626,0.013322,0.0,0.0,0.0,0.519333,1.272686,0.0,0.0,0.0,0.532389,0.0,0.550662,0.263516,0.033515,0.611871,0.880727,0.304514,1.195917,0.129151,0.04601,0.0,1.219458,0.210007,1.754441,0.529761,0
"""/storage/home/hcoda1/7/acheung…",3.0,6.0,0.079557,2.158225,0.970361,0.0,1.015812,0.547688,0.341653,0.155102,0.114957,0.0,0.0,0.586919,0.155601,0.591243,0.03109,1.032308,0.790121,0.730858,0.0,1.297465,0.000998,0.125733,0.107348,0.041562,0.780195,1.038781,0.0,0.243233,0.53852,0.0,1.243877,0.86334,0.656169,1.564929,…,0.334183,0.727118,0.172298,0.058998,0.038082,0.121399,0.061113,0.379611,0.092362,0.522844,0.0,0.38986,0.0,0.0,0.050778,0.268432,0.518953,0.280309,0.0,0.191015,0.092905,0.01835,0.073994,1.36536,0.149721,0.120322,0.396275,0.507074,0.182179,0.191319,0.325938,0.0,1.940921,0.761787,0.853025,0.0,0
"""/storage/home/hcoda1/7/acheung…",4.0,7.0,0.0,2.082779,1.094857,0.0,0.778769,0.349799,0.144341,0.579971,0.013571,0.0,0.0,0.05166,0.502043,0.544351,0.0,0.909057,0.446395,0.725157,0.078493,1.50786,0.417534,0.0,0.474653,0.300856,0.715112,0.352991,0.0,1.191465,1.067876,0.223735,1.495245,0.504632,0.70172,0.885171,…,0.658137,0.0,0.119978,0.070171,0.0,0.0,0.484041,0.0,0.170101,0.103452,0.0,0.567871,0.0,0.0,0.442525,0.859334,0.196393,0.018195,0.383672,0.179479,0.08112,0.0,0.800949,0.410062,0.420333,1.004975,0.740951,0.243811,0.519912,0.418317,0.002026,0.0,1.497738,0.214118,0.992445,1.318701,1


In [137]:
df = df.group_by(
    ["file", "interval"]
).agg(
    pl.col("*").exclude(["file", "interval", "start_time", "end_time"]).mean()
)
df = df.with_columns(
    pl.col("interval")
    .add(1)
    .mul(interval_length)
    .alias("end_time")
).drop("interval")
df.head()

file,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,…,988,989,990,991,992,993,994,995,996,997,998,999,1000,1001,1002,1003,1004,1005,1006,1007,1008,1009,1010,1011,1012,1013,1014,1015,1016,1017,1018,1019,1020,1021,1022,1023,end_time
str,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,…,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,i64
"""/storage/home/hcoda1/7/acheung…",0.067312,2.551396,1.294874,0.096063,0.766588,0.572274,0.244008,0.587255,0.100497,0.009032,0.00814,0.096297,0.409135,0.41998,0.004575,1.291158,0.494307,0.717026,0.116346,0.937808,0.102771,0.274532,0.534448,0.097842,0.676447,0.420542,0.019285,0.522874,0.801309,0.009665,1.121017,0.287889,0.587904,1.130842,0.013419,0.481961,…,0.178589,0.605354,0.027188,0.074005,0.038626,0.057925,0.442488,0.325365,0.438612,0.142633,0.428118,0.509148,0.010984,0.0,0.347291,0.475852,0.336519,0.016798,0.160971,0.267972,0.286825,0.012534,0.790937,0.23662,0.031563,0.349288,0.684639,0.301539,0.358282,1.059151,0.114021,0.037666,1.42064,0.607609,0.863365,0.513562,25
"""/storage/home/hcoda1/7/acheung…",0.103568,2.189981,1.421594,0.0,0.879744,0.455558,0.121703,0.472365,0.052858,0.0,0.0,0.192949,0.165714,0.282821,0.017314,1.08242,0.391926,0.773387,0.02587,1.06541,0.123977,0.098408,0.514437,0.057105,0.718473,0.423487,0.0,0.412243,0.864251,0.242641,0.955122,0.351389,0.873296,1.267235,0.033144,0.212338,…,0.549446,0.272761,0.301396,0.033737,0.00952,0.03035,0.413855,0.135406,0.452226,0.444684,0.172937,0.173548,0.0,0.0,0.20964,0.348112,0.610153,0.105684,0.0,0.094004,0.186334,0.004588,0.9062,0.760346,0.146145,0.553253,0.799517,0.356233,0.661052,0.60109,0.097024,0.0,1.380321,0.425258,1.121056,0.726033,5
"""/storage/home/hcoda1/7/acheung…",0.016756,2.221521,0.958955,0.001577,0.851583,0.564496,0.137307,0.598302,0.086777,0.0,0.0,0.255748,0.265238,0.373978,0.1109,1.1797,0.530202,0.784626,0.18814,1.406455,0.243337,0.064144,0.376905,0.193655,0.515906,0.336983,0.007156,0.724908,0.533907,0.118971,1.223903,0.322139,1.011274,1.086738,0.0,0.328687,…,0.534914,0.232194,0.035302,0.160954,0.0,0.034025,0.270747,0.030459,0.421332,0.461571,0.129716,0.455217,0.0,0.000853,0.312928,0.612465,0.322756,0.167746,0.318211,0.16432,0.334421,0.0,0.548968,0.310739,0.302577,0.355214,0.812282,0.381214,0.531312,0.66322,0.04383,0.0,1.546287,0.492773,0.81705,1.022706,10
"""/storage/home/hcoda1/7/acheung…",0.11128,0.056397,0.267574,0.189332,0.86353,0.424066,0.061114,0.194437,0.601533,0.0,0.345025,0.082018,0.617093,0.475512,0.228849,0.266922,0.05562,0.416883,0.04452,0.405015,0.212901,0.049541,0.409871,0.348032,0.17852,0.002716,0.193395,1.112712,0.576922,0.026384,1.25183,0.03473,0.027907,0.017582,0.466891,0.265386,…,0.550613,0.179524,0.160209,0.017156,0.056307,0.582003,0.25729,0.165378,0.24753,0.084463,0.04009,0.034858,0.04932,0.171017,0.285443,0.092923,0.217692,0.422725,0.057322,0.0,0.280604,0.07862,0.861156,1.06539,0.06495,0.224339,0.98609,0.144952,0.038038,0.0,0.052551,0.24067,0.804743,0.276193,0.235538,0.721574,5
"""/storage/home/hcoda1/7/acheung…",0.0,0.215968,0.707746,0.694977,0.706525,0.256528,0.167601,0.284499,0.794868,0.041145,0.192595,0.075004,0.254766,0.07241,0.182477,0.362865,0.0,0.206099,0.323101,0.205715,0.805781,0.004924,0.716925,0.020563,0.135841,0.045427,0.332029,0.994015,0.487605,0.079549,0.360386,0.609935,0.289414,0.309157,0.026568,0.009334,…,0.504358,0.122067,0.643297,0.0,0.062802,0.130874,0.261947,0.242359,0.168348,0.36048,0.25706,0.105201,0.0,0.108803,1.081561,0.226356,0.0,0.141584,0.085847,0.011707,0.382171,0.035394,0.502948,0.256902,0.107737,0.272693,0.149034,0.453491,0.176057,0.082623,0.008077,1.052729,0.890317,0.188679,0.44152,1.46194,15


In [138]:
df = df.select(
    "file",
    "end_time",
    (
        pl.concat_list(pl.all().exclude("file", "end_time"))
        .list.to_array(len(df.columns) - 2)
        .alias("embedding")
    ),
).sort("file", "end_time")
X = df.get_column("embedding").to_torch().to(torch.float32)
with torch.no_grad():
    pred = torch.softmax(classifier(X), dim=1)
df = df.with_columns(pl.Series("predictions", pred.numpy().tolist()))

df

file,end_time,embedding,predictions
str,i64,"array[f32, 1024]",list[f64]
"""/storage/home/hcoda1/7/acheung…",5,"[0.103568, 2.189981, … 0.726033]","[5.0815e-22, 1.5232e-21, … 4.5810e-8]"
"""/storage/home/hcoda1/7/acheung…",10,"[0.016756, 2.221521, … 1.022706]","[4.0815e-24, 3.3824e-23, … 3.6340e-9]"
"""/storage/home/hcoda1/7/acheung…",15,"[0.007265, 2.535607, … 0.705621]","[1.0775e-17, 3.0516e-17, … 2.1201e-9]"
"""/storage/home/hcoda1/7/acheung…",20,"[0.012647, 2.096784, … 0.66197]","[1.2913e-22, 1.6474e-20, … 7.7085e-10]"
"""/storage/home/hcoda1/7/acheung…",25,"[0.067312, 2.551396, … 0.513562]","[1.6528e-19, 8.8323e-20, … 2.6155e-7]"
"""/storage/home/hcoda1/7/acheung…",30,"[0.0, 2.141247, … 0.903159]","[1.7283e-20, 8.0578e-18, … 0.000003]"
"""/storage/home/hcoda1/7/acheung…",5,"[0.11128, 0.056397, … 0.721574]","[1.9022e-13, 1.2241e-11, … 0.000221]"
"""/storage/home/hcoda1/7/acheung…",10,"[0.026202, 0.062568, … 1.27204]","[4.0099e-11, 1.0526e-7, … 8.1828e-9]"
"""/storage/home/hcoda1/7/acheung…",15,"[0.0, 0.215968, … 1.46194]","[2.0014e-11, 5.0843e-11, … 3.5300e-9]"
